# Tez: ask questions about your Excel or CSV table in Uzbek, Russian or English

Tez is a small AI model (4B parameters) that reads a request about a table and writes a short **plan**, for example "group by branch, then sum the amount". It never calculates numbers itself. **DuckDB** runs the plan on the data, so the numbers come from the data, not from the model.

```
your request  ->  Tez model writes a plan  ->  DuckDB runs the plan  ->  result table
```

When a request is unclear, for example "analyse it", Tez asks one short question instead of guessing.

**What this notebook does**

1. Installs llama.cpp and downloads the model (about 3 GB, a few minutes).
2. Shows Tez on a small demo sales table, in four languages.
3. Lets you upload **your own** CSV or Excel file and ask about it.

Everything runs inside this Colab machine. A CPU runtime is enough, and no GPU is needed.

Run the cells from top to bottom with **Shift + Enter**.

## Step 1. Install llama.cpp

llama.cpp is the program that runs the model. This downloads the ready-made Linux build that the benchmark used (b11946) and installs DuckDB.

In [ ]:
!wget -q https://github.com/ggml-org/llama.cpp/releases/download/b11946/llama-b11946-bin-ubuntu-x64.tar.gz
!tar -xzf llama-b11946-bin-ubuntu-x64.tar.gz
!pip install -q duckdb openpyxl

## Step 2. Download the model and the Tez code

This downloads the whole repository:

- the model file;
- the prompt and tool list (`prompt/`);
- the table describer (`training/make_table.py`);
- the plan runner (`accuracy/tools.py`).

In [ ]:
from huggingface_hub import snapshot_download

repo = snapshot_download("Abulqosim0227/Tez-Qwen3.5-4B-GGUF")
print(repo)

## Step 3. Start the model server

This starts llama.cpp in the background on this Colab machine. The cell waits until the model is loaded, about a minute.

In [ ]:
import subprocess
import time
import urllib.request

SERVER = "http://127.0.0.1:8080"
server = subprocess.Popen(
    ["llama-b11946/llama-server", "-m", f"{repo}/Tez-tez17-agent-Q4_K_M.gguf", "--jinja", "-c", "8192",
     "--host", "127.0.0.1", "--port", "8080"],
    stdout=open("server.log", "w"), stderr=subprocess.STDOUT)
while server.poll() is None:
    try:
        urllib.request.urlopen(f"{SERVER}/health")
        print("model ready")
        break
    except OSError:
        time.sleep(2)
else:
    print(open("server.log").read()[-2000:])

## Step 4. The helper functions

You don't need to change this cell. It defines three functions you will use:

- **`open_table(path)`** opens a CSV or Excel file and shows what the model will see: each column's name, type, and its listed values or a few samples. Only this description goes to the model, not the data rows.
- **`ask(request)`** sends your request to the model, shows the plan it wrote, runs the plan with DuckDB, and shows the result. If the model asks a question instead, you see the question.
- **`plan(request)`** returns only the plan, without running it.

Each request works on the whole table. Follow-up requests that build on the previous result are a feature of the Tez app and are not included here.

In [ ]:
import json
import sys
from pathlib import Path

import duckdb
import pandas as pd
from IPython.display import Markdown, display

sys.path.insert(0, repo)
from prompt import build
from accuracy.tools import ROW_ID, TOOLS, ToolError, q
from training.make_table import describe_table

SETTINGS = {**json.loads(Path(repo, "accuracy/settings.json").read_text(encoding="utf-8")), "today": ""}
PROFILE = json.loads(Path(repo, "training/describe.json").read_text(encoding="utf-8"))
CHARTS = {"line": "line", "pie": "pie"}
session = {}


def open_table(path, name=None, glossary=None, examples=None):
    source = str(path)
    if source.lower().endswith((".xlsx", ".xls")):
        source = f"{source}.csv"
        pd.read_excel(path).to_csv(source, index=False)
    con = duckdb.connect()
    con.execute(f"CREATE TABLE base AS SELECT row_number() OVER () AS {q(ROW_ID)}, * FROM read_csv(?)", [source])
    con.execute(f"CREATE VIEW described AS SELECT * EXCLUDE ({q(ROW_ID)}) FROM base")
    table = describe_table(con, "described", name or Path(path).stem, PROFILE["type_names"], PROFILE["samples_per_column"],
                           PROFILE["values_limit"], PROFILE["values_chars_limit"])
    session.update(con=con, context=build.make_context(table, glossary or {}, examples or []))
    display(Markdown(f"**{table['name']}**: {table['rows']:,} rows, {len(table['columns'])} columns"))
    return pd.DataFrame([{"column": column["name"], "type": column["type"], "shown to the model": column.get("values", column.get("samples"))}
                         for column in table["columns"]])


def plan(request):
    context = session["context"]
    body = {
        "messages": build.messages(context, request),
        "temperature": 0,
        "max_tokens": 200,
        "chat_template_kwargs": {"enable_thinking": False},
        "response_format": {"type": "json_schema", "json_schema": {"name": "answer", "schema": build.output_schema(context)}},
    }
    call = urllib.request.Request(f"{SERVER}/v1/chat/completions", json.dumps(body).encode("utf-8"), {"Content-Type": "application/json"})
    with urllib.request.urlopen(call) as response:
        return json.loads(json.load(response)["choices"][0]["message"]["content"])


def run(steps):
    con, current, made, notes = session["con"], "base", [], []
    try:
        for index, step in enumerate(steps, 1):
            target = f"step_{index}"
            notes += TOOLS[step["tool"]](con, current, target, step["args"], SETTINGS)
            made.append(target)
            current = target
        return con.execute(f"SELECT * FROM {q(current)}").df().drop(columns=[ROW_ID], errors="ignore"), notes
    finally:
        for table in made:
            con.execute(f"DROP TABLE IF EXISTS {q(table)}")


def show(result, notes):
    for note in notes:
        if "chart" in note:
            data = result.set_index(note["x"])[note["y"]]
            axis = data.plot(kind=CHARTS.get(note["chart"], "bar"), title=f"{note['y']} by {note['x']}", figsize=(8, 4), rot=0)
            axis.figure.tight_layout()
        if "export" in note:
            name = f"natija.{note['export']}"
            result.to_excel(name, index=False) if note["export"] == "xlsx" else result.to_html(name, index=False)
            display(Markdown(f"Saved as `{name}` in the Colab files panel."))
    display(result)


def ask(request):
    answer = plan(request)
    if "ask" in answer:
        display(Markdown(f"**Tez asks:** {answer['ask']}"))
        return
    display(Markdown("**Plan:** " + " -> ".join(f"`{step['tool']}` {json.dumps(step['args'], ensure_ascii=False)}" for step in answer["steps"])))
    try:
        show(*run(answer["steps"]))
    except (ToolError, duckdb.Error) as error:
        display(Markdown(f"**The plan could not run:** {error}"))

## Step 5. Try it on a demo table

This makes a small sales table with made-up data: 600 sales, 5 branches and 5 products, during 2026. Then `open_table` shows what the model sees.

In [ ]:
import random
from datetime import date, timedelta

random.seed(7)
products = {"Non": 4000, "Sut": 12000, "Guruch": 18000, "Yog'": 25000, "Shakar": 14000}
rows = []
for number in range(1, 601):
    product = random.choice(list(products))
    count = random.randint(1, 40)
    rows.append({"savdo_id": number, "sana": date(2026, 1, 1) + timedelta(days=random.randint(0, 300)),
                 "filial": random.choice(["Toshkent", "Samarqand", "Buxoro", "Andijon", "Namangan"]),
                 "mahsulot": product, "soni": count, "narx": products[product], "summa": count * products[product]})
pd.DataFrame(rows).to_csv("savdo.csv", index=False)
open_table("savdo.csv")

### Requests in four languages

The first answer takes about half a minute, because the model reads the table description for the first time. After that, each answer takes a few seconds.

In [ ]:
ask("Filiallar bo'yicha jami savdo summasi")

In [ ]:
ask("Энг кўп сотилган 3 та маҳсулотни кўрсат")

In [ ]:
ask("Покажи продажи в Самарканде на сумму больше 500 000")

In [ ]:
ask("Average price by product")

In [ ]:
ask("Mahsulotlar bo'yicha savdo summasini diagramma qilib ko'rsat")

### An unclear request

"Tahlil qil" means "analyse it". It names no action and no column, so Tez asks what you want instead of guessing.

In [ ]:
ask("Tahlil qil")

## Step 6. Use your own file

Run the next cell and pick a CSV or Excel file from your computer. The first row must hold the column names.

The file is uploaded to this Colab machine, not to Hugging Face. Don't upload data that you aren't allowed to put on Google's servers. For private data, use the offline Tez app instead.

In [ ]:
from google.colab import files

uploaded = files.upload()
my_file = next(iter(uploaded))
open_table(my_file)

In [ ]:
ask("Write your request here")

### Teach it your business words (optional)

If your columns have short or coded names, a small glossary helps. It is a dictionary of words, with what each one means in your table:

```python
open_table(my_file, glossary={
    "tushum / выручка / revenue": "column summa",
    "faol / активные / active": "holat = Faol",
})
```

You can also pass two or three `examples`, in the same format as `examples/bank_loans/examples.json` in the repository.

## The 18 tools the model can use

| Tool | What it does |
|---|---|
| `filter`, `remove_rows` | keep or remove rows by a condition |
| `filter_dates` | keep a period, such as last month or this year |
| `group_and_total` | totals, averages, counts, minimum or maximum, overall or by groups |
| `top_n` | the largest or smallest N rows or groups |
| `period_compare` | compare a period with the previous one |
| `sort`, `select_columns` | order rows, keep chosen columns |
| `add_computed_column`, `add_value_column` | difference, ratio, sum, product or percent of columns |
| `make_chart` | a chart of the result |
| `fix_dates`, `fix_numbers`, `unify_spelling`, `remove_duplicates`, `fill_empty` | clean the data |
| `export_xlsx`, `export_html` | save the result |

A plan has at most 4 steps.

## Good to know

- **Check the results.** This notebook runs plans directly. The Tez app also checks each result before showing it, for example an empty result or a filter that kept every row, and asks first before steps that change values. Those checks are not included here.
- **Comparing two columns is weak.** For "rows where loans exceed deposits", ask for the difference column first, then filter on it.
- **Values must match the data.** For text columns with few values, the model can only use values that exist in the table.
- **Results on the internal benchmark:** 92.9% of clear requests correct, model alone. On a harder retail test the result was much weaker. See the model card for details and limits.
- **If a cell hangs or fails,** read `server.log` in the Colab files panel.

## Stop the server

Run this when you are done, to free the memory.

In [ ]:
server.terminate()